<a href="https://colab.research.google.com/github/oleg456-g/krasniy/blob/main/Untitled4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers==4.51.3 trl==0.17.0 peft==0.15.2 bitsandbytes==0.46.1 datasets==3.5.0 accelerate==1.6.0 scikit-learn==1.7.2

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
set_seed(42)
id_model = "Qwen/Qwen3-4B-Instruct-2507"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(id_model)
model = AutoModelForCausalLM.from_pretrained(id_model, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [3]:
import json
from datasets import Dataset
from peft import prepare_model_for_kbit_training
rows = [json.loads(l) for l in open("kid_adult.jsonl", encoding="utf-8")]

train = Dataset.from_list([
    {"messages": [
        {"role": "user", "content": r["prompt"]},
        {"role": "assistant", "content": r["kid"]},
    ]}
    for r in rows
])

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

In [4]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

sft_cfg = SFTConfig(
    output_dir="sft_out",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=512,
    optim="paged_adamw_8bit",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="no",
    seed=42,
    report_to="none")

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM")

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train,
    peft_config=peft_cfg,
    processing_class=tok)
trainer.train()
trainer.model.save_pretrained("adapters/task1_sft")

Converting train dataset to ChatML:   0%|          | 0/1489 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,2.123400
20,1.441000
30,1.333900
40,1.222700
50,1.216700
60,1.178700
70,1.190300
80,1.170700
90,1.193800


In [7]:
import json
import pickle
from scipy.sparse import hstack

test = [json.loads(l) for l in open("public_test_style.jsonl", encoding="utf-8")]
trainer.model.save_pretrained("adapters/task1_sft")
model.config.use_cache = True

model.eval()
gens = []

for r in test:
    text = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True,)
    enc = tok(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=False)
    gens.append(tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True))

d = pickle.load(open("style_clf.pkl", "rb"))
clf, (v1, v2) = d["clf"], d["vecs"]
X = hstack([v1.transform(gens), v2.transform(gens)])
p = clf.predict_proba(X)[:, 1]
print(f"Средний P_simple, SFT: {p.mean():.4f}")

Средний P_simple, SFT: 0.9775


In [9]:
from trl import DPOConfig, DPOTrainer
from peft import PeftModel
import json

if not isinstance(model, PeftModel):
    model = PeftModel.from_pretrained(model, "adapters/task1_sft", is_trainable=True)

model.eval()
model.config.use_cache = True

dpo_ds = Dataset.from_list([
    {
        "prompt":   [{"role": "user", "content": r["prompt"]}],
        "chosen":   [{"role": "assistant", "content": r["kid"]}],
        "rejected": [{"role": "assistant", "content": r["adult"]}],
    }
    for r in rows
])

model.config.use_cache = False

dpo_cfg = DPOConfig(
    output_dir="dpo_style_out",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    optim="paged_adamw_8bit",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="no",
    seed=42,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    args=dpo_cfg,
    train_dataset=dpo_ds,
    processing_class=tok,
)

trainer.train()
trainer.model.save_pretrained("adapters/task2_dpo_style")

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Адаптер подгружен с диска


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_

КОНТРОЛЬ: Продавцы делают скидки, чтобы быстро продать старые игрушки или одежду, освободить место для новых красивых вещей. Это как если бы ты отдал старую игрушку брату, чтобы купить новую. А ещё они так делают, чтобы привлечь больше друзей и купить у них что-то вкусное! 



Extracting prompt in train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
10,0.169700
20,0.028100
30,0.053400
40,0.024200
50,0.012200
60,0.011600
70,0.005600
80,0.037300
90,0.094900


Сохранено: adapters/task2_dpo_style


In [10]:
import json
import pickle
from scipy.sparse import hstack

test = [json.loads(l) for l in open("public_test_style.jsonl", encoding="utf-8")]
model.config.use_cache = True

model.eval()
gens = []

for r in test:
    text = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True,)
    enc = tok(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=False)
    gens.append(tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True))

d = pickle.load(open("style_clf.pkl", "rb"))
clf, (v1, v2) = d["clf"], d["vecs"]
X = hstack([v1.transform(gens), v2.transform(gens)])
p = clf.predict_proba(X)[:, 1]
print(p.round(2))
print(f"Средний P_simple, DPO: {p.mean():.4f}")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_

[1.   0.96 0.99 1.   0.99 1.   1.   1.   0.99 1.   0.98 0.98 0.99 0.97
 0.97 0.99 0.99 0.99 1.   1.   0.99 1.   0.99 0.99 0.98 1.   1.   1.
 1.   1.   1.   0.99 1.   1.   1.   1.   0.99 1.   1.   0.99 1.   1.
 1.   1.   0.99 0.99 1.   0.99 0.99 0.99]
Средний P_simple, DPO: 0.9924
